# Signal Hunters R2 - run 13 (teammate C): FIXED + 2x canvas + NEW pseudo-labels (6598) - EfficientNet-B2 + B0
GPU T4 x2, Persistence: Files only, Internet ON, input: signal-hunters-r2.

In [ ]:
%%writefile /kaggle/working/cnn_r2.py
"""Signal Hunters R2 - CNN on the 'global-time canvas'.
Each cadence -> one 128x128 image: the 6 panels placed on ONE time axis including the slew gaps
(panel p row r at t = p*(16+gap)+r), so a true ETI drift is a single straight line through the 3 ON panels.
Input channels: z (bandpass-flattened, noise-normalised), zr (z minus per-row median: kills broadband rows),
on/off indicator (+1 ON rows, -1 OFF rows, 0 gaps), valid mask.
Domain shift GBT->PKS is attacked with augmentation: frequency zoom (PKS drifts look ~0.64x as steep),
SNR loss, dropped rows, broadband glitch rows, per-panel gain, frequency shift/flip.
Usage: python cnn_r2.py DATA_DIR EPOCHS SEEDS [arch=res|timm_resnet18|timm_effb0] [PL]
Writes oof_<tag>.npy (18000x9) and test_<tag>.npy (10000x9) to ./out/"""
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn, sys, os, time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
LABELS = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass', 'radar_chirp',
          'rfi_intermittent', 'rfi_persistent', 'noise']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
UP = int(os.environ.get('UP', 1)); P_ETI = float(os.environ.get('P_ETI', 0)); P_TRAP = float(os.environ.get('P_TRAP', 0))

def normalise(X):
    x = X.astype(np.float32); m = (X > 0); x[~m] = np.nan
    x = x - np.nanmedian(x, axis=2, keepdims=True)
    s = 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x, axis=(2, 3), keepdims=True)), axis=(2, 3), keepdims=True) + 1e-3
    z = np.nan_to_num(np.clip(x / s, -10, 60))
    return z.astype(np.float16), m

def load(D, split):
    X = np.load(f'{D}/{split}_cadences.npy'); meta = pd.read_csv(f'{D}/{split}_meta.csv')
    return X, (X > 0), meta.slew_gap_rows.values.astype(np.int64), meta

def norm_t(x, m):
    """FIXED normalisation (v2). Old version subtracted each channel's median over time, which ERASED every
    non-drifting line (persistent RFI, non-drifting traps): a 10-sigma persistent line became 0.4 sigma.
    Now: bandpass = per-panel time-median spectrum smoothed ACROSS FREQUENCY (median, 9 channels), so narrow
    lines survive. Channel 64 (GBT-only centre spike, absent in PKS) is blanked in every cadence."""
    x = torch.where(m > 0, x, torch.full_like(x, float('nan')))
    x[..., 64] = float('nan')
    spec = torch.nanmedian(x, dim=2).values                                   # (B,6,128)
    spec = torch.where(torch.isnan(spec), torch.nanmedian(spec, dim=-1, keepdim=True).values, spec)
    pad = Fn.pad(spec, (4, 4), mode='replicate')
    band = pad.unfold(-1, 9, 1).median(-1).values                            # smooth bandpass
    r = x - band[:, :, None, :]
    f = r.flatten(2); med = torch.nanmedian(f, dim=2, keepdim=True).values
    s = 1.4826 * torch.nanmedian((f - med).abs(), dim=2, keepdim=True).values + 1e-3
    z = ((r - med[..., None]) / s[..., None]).clamp(-10, 60)
    return torch.nan_to_num(z, nan=0.0)

def byte_aug(x, m, k_lo=5., k_hi=30., p=0.8):
    """PKS-like receiver noise in the POWER domain: power *= Gamma(k)/k  -> skewed log-noise like the test
    telescope (measured: noise MAD 15.6 vs 11.9 bytes, more negative skew). Also extra dropped/glitch rows."""
    B = x.shape[0]
    k = torch.empty(B, 1, 1, 1, device=x.device).uniform_(k_lo, k_hi)
    g = torch.distributions.Gamma(k.expand_as(x), k.expand_as(x)).sample().clamp_min(1e-4)
    sel = (torch.rand(B, 1, 1, 1, device=x.device) < p).float()
    x = x + sel * 24 * torch.log2(g)
    drop = (torch.rand(B, 6, 16, 1, device=x.device) < torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, 0.03)).float()
    gl = (torch.rand(B, 6, 16, 1, device=x.device) < 0.01).float() * torch.empty(B, 6, 16, 1, device=x.device).uniform_(10, 50)
    x = (x + gl).clamp(1, 255)
    m = m * (1 - drop)
    return x, m

# ---------------- GPU-side augmentation + canvas -----------------
def augment(z, m, train):
    """z,m: (B,6,16,128) float on device"""
    B = z.shape[0]
    if not train: return z, m
    # SNR loss + extra noise (PKS noisier): z' = a*z + sqrt(1-a^2)*N
    a = torch.empty(B, 1, 1, 1, device=z.device).uniform_(0.75, 1.0)
    z = a * z + torch.sqrt(1 - a ** 2) * torch.randn_like(z)
    # per-panel gain on the signal part
    z = z * torch.empty(B, 6, 1, 1, device=z.device).uniform_(0.8, 1.2)
    # frequency zoom (compress/expand channel axis), p=0.7, s in [0.55,1.1]
    sel = torch.rand(B, device=z.device) < 0.15
    if sel.any():
        s = torch.ones(B, device=z.device); s[sel] = torch.empty(int(sel.sum()), device=z.device).uniform_(0.85, 1.15)
        xs = torch.linspace(-1, 1, 128, device=z.device)
        gx = xs[None, :] / s[:, None]                                   # sample positions in input
        grid = torch.stack([gx[:, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=z.device)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, mode='bilinear', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        mm = Fn.grid_sample(m.reshape(B, 1, 96, 128).float(), grid, mode='nearest', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        outside = (gx.abs() > 1)[:, None, None, :].expand(B, 6, 16, 128)
        # bilinear interpolation shrinks noise std -> renormalise per panel, fill outside with fresh noise
        zz = torch.where(outside, torch.randn_like(zz), zz)
        mm = torch.where(outside, torch.ones_like(mm), mm)
        sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        z = torch.where(sel[:, None, None, None], zz, z); m = torch.where(sel[:, None, None, None], mm, m.float())
    m = m.float()
    # frequency shift (roll) and flip
    sh = int(np.random.randint(-30, 31)); z = torch.roll(z, sh, -1); m = torch.roll(m, sh, -1)
    fl = torch.rand(B, device=z.device) < 0.5
    z = torch.where(fl[:, None, None, None], z.flip(-1), z); m = torch.where(fl[:, None, None, None], m.flip(-1), m)
    return z, m

def up(C):
    return Fn.interpolate(C, scale_factor=UP, mode='nearest') if UP > 1 else C

def canvas(z, m, gap):
    """(B,6,16,128) -> (B,4,128,128) global-time canvas"""
    B = z.shape[0]; H = 128
    zr = z - z.median(-1, keepdim=True).values
    C = torch.zeros(B, 4, H, 128, device=z.device)
    p = torch.arange(6, device=z.device); r = torch.arange(16, device=z.device)
    t = p[None, :, None] * (16 + gap[:, None, None]) + r[None, None, :]          # (B,6,16)
    t = t + ((H - (6 * 16 + 5 * gap)) // 2)[:, None, None]                       # centre vertically
    bi = torch.arange(B, device=z.device)[:, None, None].expand(B, 6, 16)
    sign = torch.tensor([1., -1, 1, -1, 1, -1], device=z.device)[None, :, None].expand(B, 6, 16)
    C[bi, 0, t] = z * m; C[bi, 1, t] = zr * m
    C[bi, 2, t] = sign[..., None].expand(B, 6, 16, 128); C[bi, 3, t] = m
    return C


def inject(z, m, gap, yb, p_eti, p_trap):
    """Physics-based synthetic signals (labels follow the priority order):
    - ETI: a narrow drifting line on ONE straight track in global time, ON panels only -> label eti_candidate
    - trap: ON-only lines that do NOT connect (random offsets) or do not drift -> rfi_intermittent
      (only injected into noise / rfi_persistent cadences, whose priority is lower)"""
    B = z.shape[0]; d_ = z.device
    t = (torch.arange(6, device=d_)[None, :, None] * (16 + gap[:, None, None]) + torch.arange(16, device=d_)[None, None, :]).float()
    tc = t[:, [0, 2, 4]].mean((1, 2))[:, None, None]
    ch = torch.arange(128, device=d_).float()[None, None, None, :]
    on = torch.tensor([1., 0, 1, 0, 1, 0], device=d_)[None, :, None, None]
    r = torch.rand(B, device=d_)
    e = (r < p_eti) & (yb != 0)
    tr_ = (r >= p_eti) & (r < p_eti + p_trap) & ((yb == 7) | (yb == 8))
    if e.any() or tr_.any():
        drift = torch.empty(B, device=d_).uniform_(0.15, 1.3) * (torch.randint(0, 2, (B,), device=d_) * 2 - 1)
        c0 = torch.empty(B, device=d_).uniform_(20, 108)
        amp = torch.empty(B, device=d_).uniform_(0.6, 3.0); wid = torch.empty(B, device=d_).uniform_(0.5, 1.2)
        centre = c0[:, None, None] + drift[:, None, None] * (t - tc)                      # (B,6,16) connected track
        kind = torch.randint(0, 2, (B,), device=d_)                                       # trap type
        off = torch.empty(B, 6, 1, device=d_).uniform_(-40, 40)                           # disconnect offsets
        tr_centre = torch.where(kind[:, None, None] == 0, c0[:, None, None] + off + drift[:, None, None] * (t - t[:, :, :1]),
                                c0[:, None, None] + 0 * t)                                # non-connecting / non-drifting
        cen = torch.where(e[:, None, None], centre, tr_centre)
        line = amp[:, None, None, None] * torch.exp(-0.5 * ((ch - cen[..., None]) / wid[:, None, None, None]) ** 2) * on
        sel = (e | tr_)[:, None, None, None].float()
        z = z + sel * line * m
        yb = torch.where(e, torch.zeros_like(yb), torch.where(tr_, torch.full_like(yb, 6), yb))
    return z, yb

# ---------------- model -----------------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o)) if (i != o or st != 1) else nn.Identity()
    def forward(s, x): return torch.relu(s.b2(s.c2(torch.relu(s.b1(s.c1(x))))) + s.sk(x))

class ResNet(nn.Module):
    def __init__(s, w=40):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(4, w, (5, 3), 1, (2, 1), bias=False), nn.BatchNorm2d(w), nn.ReLU())
        s.body = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1),
                               Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1), Block(4 * w, 6 * w, 2), Block(6 * w, 6 * w, 1),
                               Block(6 * w, 8 * w, 2))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(16 * w, 9))
    def forward(s, x):
        h = s.body(s.stem(x)); return s.head(torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1))

def make(arch):
    if arch == 'res': return ResNet()
    import timm
    name = {'timm_resnet18': 'resnet18', 'timm_effb0': 'efficientnet_b0', 'timm_effb2': 'efficientnet_b2', 'timm_convnext': 'convnext_nano', 'timm_effb3': 'efficientnet_b3'}[arch]
    try: return timm.create_model(name, pretrained=True, in_chans=4, num_classes=9)
    except Exception as e: print('pretrained download failed, random init:', e); return timm.create_model(name, pretrained=False, in_chans=4, num_classes=9)

def predict(net, Z, M, G, tta=True):
    net.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for j in range(0, len(Z), 512):
            m = torch.from_numpy(M[j:j + 512]).to(dev).float()
            z = torch.from_numpy(Z[j:j + 512]).to(dev).float()
            if Z.dtype == np.uint8: z = norm_t(z, m)
            g = torch.from_numpy(G[j:j + 512]).to(dev)
            p = torch.softmax(net(up(canvas(z, m, g))).float(), 1)
            if tta: p = 0.5 * (p + torch.softmax(net(up(canvas(z.flip(-1), m.flip(-1), g))).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)

def pks_like(Z, M, seed=0):
    """deterministic 'PKS-like' version of held-out GBT data for shift-aware validation"""
    torch.manual_seed(seed); out_z, out_m = [], []
    for j in range(0, len(Z), 512):
        x = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float(); B = len(x)
        x, m = byte_aug(x, m, 12., 12., 1.0); z = norm_t(x, m)
        xs = torch.linspace(-1, 1, 128, device=dev) / 1.0
        grid = torch.stack([xs[None, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=dev)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, align_corners=True).reshape(B, 6, 16, 128)
        out = (xs.abs() > 1)[None, None, None, :].expand_as(zz)
        zz = torch.where(out, torch.randn_like(zz), zz); sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        mm = torch.where(out, torch.ones_like(m), Fn.grid_sample(m.reshape(B, 1, 96, 128), grid, mode='nearest', align_corners=True).reshape(B, 6, 16, 128))
        out_z.append(zz.cpu().numpy().astype(np.float16)); out_m.append(mm.cpu().numpy() > 0.5)
    return np.concatenate(out_z), np.concatenate(out_m)

def fit(Z, M, G, y, w_cls, epochs, seed, arch, evals):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make(arch).to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=0.02)
    bs = int(os.environ.get('BS', 128)); n = len(y); steps = epochs * (n // bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(w_cls, device=dev, dtype=torch.float32), label_smoothing=0.05)
    scaler = torch.amp.GradScaler(enabled=dev == 'cuda'); Y = torch.from_numpy(y)
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(n); t0 = time.time(); L = 0
        for j in range(0, n - bs + 1, bs):
            idx = perm[j:j + bs]
            x = torch.from_numpy(Z[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev).float()
            g = torch.from_numpy(G[idx]).to(dev); yb = Y[idx].to(dev)
            x, m = byte_aug(x, m); z = norm_t(x, m)
            if P_ETI + P_TRAP > 0: z, yb = inject(z, m, g, yb, P_ETI, P_TRAP)
            z, m = augment(z, m, True)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(net(up(canvas(z, m, g))), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); L += loss.item() * bs
        if ep % 5 == 4 or ep == epochs - 1: print(f'   ep{ep} loss {L / n:.3f} {time.time() - t0:.1f}s', flush=True)
    return [predict(net, *e) for e in evals]

if __name__ == '__main__':
    D, EP, SEEDS = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
    arch = sys.argv[4] if len(sys.argv) > 4 else 'res'; PL = 'PL' in sys.argv
    tag = os.environ.get('TAG', arch); SMOKE = int(os.environ.get('SMOKE', 0)); os.makedirs('out', exist_ok=True)
    Z, M, G, meta = load(D, 'train'); Zt, Mt, Gt, tmeta = load(D, 'test')
    y = meta.label.map({l: i for i, l in enumerate(LABELS)}).values
    if SMOKE: Z, M, G, y = Z[:SMOKE], M[:SMOKE], G[:SMOKE], y[:SMOKE]; Zt, Mt, Gt = Zt[:SMOKE], Mt[:SMOKE], Gt[:SMOKE]
    cnt = np.bincount(y, minlength=9); w_cls = (cnt.sum() / (9 * np.maximum(cnt, 1))) ** 0.5; w_cls = w_cls / w_cls.mean()
    print('device', dev, 'arch', arch, 'train', Z.shape, 'class weights', w_cls.round(2), flush=True)
    if PL:
        pl = np.load('out/pl.npz'); sel, ypl = pl['sel'], pl['y']
        print('pseudo-labelled test cadences', sel.sum(), flush=True)
    oof = np.zeros((len(y), 8 + 1)); oof_pks = np.zeros_like(oof); pt = np.zeros((len(Zt), 9))
    for s in range(SEEDS):
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Z, y)):
            Za, Ma, Ga, ya = Z[a], M[a], G[a], y[a]
            if PL: Za, Ma, Ga, ya = np.concatenate([Za, Zt[sel]]), np.concatenate([Ma, Mt[sel]]), np.r_[Ga, Gt[sel]], np.r_[ya, ypl]
            Zp, Mp = pks_like(Z[b], M[b], seed=k)
            pb, pp, pte = fit(Za, Ma, Ga, ya, w_cls, EP, 100 * s + k + int(os.environ.get('SEED_OFF', 0)), arch, [(Z[b], M[b], G[b]), (Zp, Mp, G[b]), (Zt, Mt, Gt)])
            oof[b] += pb / SEEDS; oof_pks[b] += pp / SEEDS; pt += pte / (5 * SEEDS)
            print(f'seed {s} fold {k}: F1 {f1_score(y[b], pb.argmax(1), average="macro"):.4f} | PKS-like F1 {f1_score(y[b], pp.argmax(1), average="macro"):.4f}', flush=True)
            np.save(f'out/oof_{tag}_partial.npy', oof); np.save(f'out/test_{tag}_partial.npy', pt); np.save(f'out/folds_done_{tag}.npy', np.array([s, k]))
        f = SEEDS / (s + 1)
        print(f'== after seed {s}: OOF F1 {f1_score(y, (oof * f).argmax(1), average="macro"):.4f} | PKS-like {f1_score(y, (oof_pks * f).argmax(1), average="macro"):.4f}', flush=True)
        print('per class (PKS-like):', dict(zip(LABELS, f1_score(y, (oof_pks * f).argmax(1), average=None).round(3))), flush=True)
        np.save(f'out/oof_{tag}.npy', oof * f); np.save(f'out/oofpks_{tag}.npy', oof_pks * f); np.save(f'out/test_{tag}.npy', pt * f)
    print('DONE', flush=True)


In [ ]:
import base64, os
os.makedirs('/kaggle/working/out', exist_ok=True)
open('/kaggle/working/out/pl.npz','wb').write(base64.b64decode('UEsDBC0AAAAIAAAAIQArq2yp//////////8HABQAc2VsLm5weQEAEACQJwAAAAAAAMgGAAAAAAAAnZexcmRVDESllK9wNlDlgE35ADIoEgIiyrCmCCiWsrdIgK/ghxk8T92n9YaEqd3x+L57pVZ3S3f899fffvXNd12/1x+X98+vP75cvni4/PnDu8vjw+WnDy8fX55+/f7Dy/vnf9e/fPrl9fm6/vrz02/P198/fff59fX42ePDXw//8/VJX191vL39vL7fPtZt+baI1+3R8a6nt1/q9uS2UF0K3Q6m84hY2vL2sfwg807c8uY6TpRAHb8IPDAetVVHeZOygNTZXRFh3nIdNZYXJg8oFF/IGciOIoohzceCc4SfbVKiyX/tf3UnpjA4xVkCFNM4a3ZZOaL33W2CO1VbXpEkZATBNQRk6PbhZLd0XHLZNmFKhQ/CbIv0Q9ZE2gqqSN5mkOySMBmyFXaZzEps8LTsT5MN0yATkEASxTB66AIt0QylGmAbt1SjwGnXqAch6CKgB07+6iavqE8iQVDqkTEGBAzfOtoHtg52coDAS1PesFz3Dwyk6XnDci00Fv8DHRpmyoNKYw4H9kGSB4zbQlTB/WFSKhEa2xJLHL59kGtXI2XGrHPMofOM7sF3d8OSlr0TMAuleQZb/fK2VW1IU7EPVuBTmdJJkzLT1JlaJlNlZqEiBHzlUYcZ6+pjHltQX+aeMUqO6Rn9bFA2IQTmSIrJHNNjHS75h3MlbVa5otIm8ljuzsloDtRw+CrZommnGk21gG8pwcDOTt5ECqVrxiCrfQLW4b5tDjC0oGJiTWN7RCxX4S0TokBNO38OmrKFVpjB55EM5B6YYccJUTMi4BT02xwtQ2EbcQJV4jbnkMlfROFLF4Phe2eHdlrx0AJKbuKpXMdqjBshZjp5QZvEzwhuaW2iOxkUXD8dm7mHKBMwvoUHGU3hjNJIKASoo1lcKvw+R1wXoGXxJHkst8k3QFdR5pxlyd7ixsIBH/mdM4YIizhqERcmNdmUY0PE7DY68zQiMQmdV80U08a84lxQIUXZcbF7txFrmvZwvXA4KtC28iGrz8iIq64xYSTS4yjEKIZTbfgFKRBR1o4lRxg0ZkwQasg6NY9HNkjTIS1FrhUBhFqNSBauCV2caE4B0EyaYC5aGsyyUhhahONqcF3uBTaPT869QHvAMRURfauY/VSS8AtPbeuggmU2l8K2YWKFYmds5xmsXUHYyZ1CFJ8FsJk3YVNwqNgyJtKZ3/QxygMsUMPLYEkr3ygwbAJXrWsDTIFQjBmmM/yRjCaC1nQwSFiw11cvXDHtBBqlAAbjDJ3Jh23rmMOLT5p8SY4BpQ85RthfpF9c8U+H2QTX2XJw1OxfqtsoeqxhF+mFOP88Gg2UcE8SM41LgMbdZZL2iB+tZKqHVRhEz9GNY6Ngfwwr+BsN7OXKtzeRFoMsxpbVgBxY38OJakK+GIHFbMRrmyAS55KgnaUkgDB5CmZlMBHyFYsmYlmapohy5YOq0Zd/DiVT+4qplcYdHi2cQ/hEALkOP0UM8L3ca3fxiCLMyDTIduNxzB4o5feKo7YDZy/mf9KuokDCIpaXjtRLfcPYfqu1A02PoNVBIrxfzbyVeXWJdbwXYoBzqwGq05FsX/JuKlJcRXVKsB7CkwcVEy7tQbjsOgMNd5N3yOqAP1kdEinTqrRjrR2Bmry5SougXUhgA/OcCphlz6AUMVnkzRBDzgVWnh8ZoU3wBEodtRxxzpOZ7JQjvPulvYU8zrgiySDw7F1rwj8zSnRPLCIZb/rD8hHR09J+Eo8PypvduARWbbN1iUXgmJ3DAVgQz35Is3ekPz7rYPG5eoCzIYUxxEZlldXsPmHa2sGGOnyZ0BRN1lkGq9EasMYef08spcTlgM3VK4Y+O5h8uxIZ6lSUgDUWZ/nQEN+wbP5VvtVli/lSCxS1lBfJZ8ocxbVQsR4BfYW6jSC2iZRf8bXJHZNT1uTqIW2+TBwmgwPDX9HfFElMqrfLp2jJonw5Z+xzNwpEWLPcupWSJNMU1sCSyDIudgybBycPwULisVkYKa+DtZpK7magudxWpZ7SWNdJykk+gy7LDB+EGYoRwJ9gQRrA5t8hh4EYN1IGpnF+nBLYEByENreMWr5QZASPae8EVwQea+Mp/FgFaHrTq6IT87YUcYLl/N73CFqxvN3FDAaOL9uEKqt4DQYLae6FHTaULpTYMpL5CvzuJytbsalXUFdY6ynogVUXCHTXPc9GP9qKHpfTQxMie4rtYqKgayJg7MCiAdbEY9KDkhjf+jK1uoIc8bv/6Mx5eMKP17010p9EnB2h3qAxTjElZaEGF4O5y3LBrUnTuoRxvl0hyjBPRF7qrnhcPlAA/x8kwdrV/wBQSwMELQAAAAgAAAAhADIc+p3//////////wUAFAB5Lm5weQEAEACwzgAAAAAAAEgSAAAAAAAApdy9rmVHEcXxbc/4QzzFZAOSU5BBxGQgEgIiZOGxQEIYjREJ4il4YYyYTv7ST6uv2cnRPmd/dFdXrVq1qu/9929+9+vf/v6j5x/PP99+/e67P75/+4s3b3/55y/ffvHm7Tffvv/7+6/++odv33/97r/f/+qrv3z37vvvv/vTV3979/35j3/2059/+cVPvnjzrzc/9PjR6+d/x6cfPj/+8Pnqw+dnHz4/z3Xn8xPcf84/z/d6Tn8/z/0s173O9X3P+f2jfPZ6Pe/zfNY+fe65rvY6v3+S877nPP8z/K5z2eVVrv8015/vP8LvfW7nWTue3/v8db3uf5Xf+3muX367/Kv3d9zn+/M8+Vvt3/OO9/iD1udjXN/nK/76fe0te9Rv+/7OX+Op3/f+/t51a5zIr2q/4kXt2PjsueYhP+y6CUe63gsva/+Ov+PSOtQutXfXt/bTOmldZJfih+Kg41j5pv7U9yp/KG4bLx1H17e4UL8Tjiovdj49r536nte5TvhYO8sv6peKw463ftvr5Ge9X/lD4ytOCv+Ky8KVFZfyz8bpuq94ID/Ruj75XfHX876v/tH16H3yo+Lryj+Nm8aX8EG4Ip5T+4gHKV/0PvGsXtf4rF0UT3pv59dz+ZnidfG7fj45V77tvOVH9aeO41V+1/uEV8I91SMdb3mx/F75pvcp7s95/arzUn7pPM/4n/ze56nO07wX/+pzxQvEF5UX6k/isfLbhWc/1F9kD/l97aQ4f3JeO59j4WzXV/xX9crC14UbsufCX/Hj2k/8ouNSHS7+1HhRHhI/kU6g/F6cKz4v/nh+f3Kd7Lj4u3jPqufFq8Vftb71U9Wj5eFad/GzxUMUT/KTrkfHrXzd8S7+Ij6svKJ6Sbxy5S/ZUXVux1s8Er5LzygOKl7F65qfapc+p+spXbHXCS+FE3pu84J40IrfjkP+Ih4jOyy+Kf1S71l5YPG9znfVBUun0O/K88LNFU/i0avOuL1PeWnxXY1PurP4g/Qn8Uzhy21+l/8J9+pH0uPEQ4THHYfyTddL+UV2E9/s9+e+J9ed37ve/b1xqv6FcP52nos/3+qp9QuNU+snvqc8K5zoej35XnlHuNvnLx3uFi+kS6gOFb4r/9QflO+ks/T7jqO8UXinfC89UTxUOtvCX+nSqo9v87veI35+yweFC5rnqpeVfxT/vU+4on5c11F+J110xf05yi9XH1F1q/hg68WFY+pXqy6oP+u50rsVL+ItKy8sftD31W/OccsjxOeW7tT1VJ9IuN1x1s9W/6Pr0XpH8+75Odb6SVdS/J1DPF/5UvWB/FP5vnZQHSt/Em87h94r/nYbDwv/z1FcWHW06gHl5fW9ntN1Xv60+IDqAuXTW/2p45AfSF8Q/1q63e249Nm4Wzzmlm+fQzxw8YFVhwtPVz9C+trqZ0qnveX54sm1s+r0paeLTwsHbvur6meu/vCyl+zx5Pel96m+rl9Xb3gpn6uf9L6lR8sO8t9VJ6hfKv+Rvy4cV94Xf1OekX/e6nfah6V4LY/u+qtv0fmfQ/lVdYb4g/x41SWqi8RrhQ+1d59zDulzi/+teZ5DuCR/l992HtJhnly31lN6kPo4T87LR7TPV34oXFj1s+rc2/p99V/0/tt10nNUp0mfWP116XqKd+Xrzqf5Q/sv+hzhjuxd/KjfnGPFg/oGypv159U3F16vvv3yW8VncV9xqv5x/Uo8oPZ5cJ3WUTxk8SbxtqWny4+F96qDlj4svlq/FE8Tv1m8W3lP/FC6qfSyhZPSk4TL6t+s+lV85bb/sfaNSOdS/El/ks7YeSuvi/8tfU39jq6T+oTiV+LpL+2X6H3F/5VvpSfLHktnEf50HLd1rXjdymvS36THKu4WbtVeq+6QzqX+tOxbv9A+u9s+cJ8vfnGLm43D1edWPtF4pJP3U3xh6V56nnC9fE1+0vGcQ3iuOlf5Yl3f9Zc+peeI16+8pLxZPabjPEdxTXxsxZ3wT/W88F/2rJ1Wv1D8Yelpws9+rvgR/5V9VC9KL1z9femOva5+d6uXrz5Cny9/X/GkOFx106rLhbvSnVYfo3lwrav8oPNZfEHjE57I75snxf9kx66r+t7q0636X3qs7rvttyy9ofZRfmicdn7ivdIBNe/yEfmh6hPZ/za+xCvl3xrn6l+sOBI+yf/Uf2ndu/xq7ePp9wu3+7xziEfrPvEG+enSsdTXlR60+t/SM4q7yktat/qd6uG1DqrP1Fdd+ryuu8V55Vfl6XMo7y5dVn4inWHtoy8vFV7ofOl/so/8bOk3eo9wVPGv93VeC/elX6h+v+XT9Uv5lXQs9UHX37cKzxYv7Ln+fq3Pqd8tnlh/LS6JH4mH630rvm/rdtUjwl3Vj4rbpU/UPueQTqR4EC4Lvzov7XuTPqX3tl+s/kHH03lKh1DdsdZ76Rqys+rs1fdZ/RHFtcYr3VXx0nmqvy6dXH2y1c9a+CweJB1i1dPC6frz0g3lV9JJpBcp3msf6XGqD5Z/K4/e1guya/Ghdlm4Kx2j6yF9eeUx9UWlt9UO0sMX711+LXvIH8QHNE/VUfJj5a9VfytOVf8qLrt+xSftU+t6yF9736p7ZEfhV+P2Frdv9YFzqD+mc+lCtafqgcaN6tr64dK7VYdJz1j6h/Bq8eg+f+kKyqd938Ltrsczfl/6Z9+3+jEdp/5e5bb/Kz5Sf1r7e4Tn6sOo/ym9ZeVJ9UPEZzRvxYV0C/Fa5XfZVflb+CPeKl1L8aw4WXghfqB+y61OJl4mnqi4EP6rDmx9tPKt4ll8TLxE8+l4lGdVRytetN9d/FB1ztLRlLdv9xEsvxbuCwdkD+Uj6Sa3+UfrJv1AvEV5U3itdZGOWX+XjtN1WPq+1rnvXf1M6YPqr9X+xdnGsfb5L72g8179q16nfVrqxynPKq81DpQvpI/XP5VXqwtJP9Kn5iN/Xv0a8V/hr/BYeLb4v+YlHWTlt7WPRzxROHcO1cF6jnTblc9WP039I9XV4pmq91d+E77q/2stHUb8a+Xz/q7rxbtUr0qXuNWT5RcrzsV313gVF6sfKvy91f2UfxTHWnf1O7qO0j+lx8g+8vMVV+VdynMr/yx9ZdVtSw/Qc+WnynPib+dYPPQcwpHbeln1g3ii+Ld4pOJh6Y8an3iQ+Lzq99s6UH7z5Fz1iNav9l77HRbPUj7S+ITDwsfG/+rjrM+uc/Pak3Plv/qn+keqj4VHyr+3/bNVD9zGt+Lq1h+6rrf/V2PpMQsfb3Wq236f+kB639Jvuo49pMd23dpfEH6JD6qulL+seBberr547f7/5lX5p8alOqjvX3xSuK14W/uDhDPC3cb/7f+Xkh1XPadxr/8zUrsKJ2/rk45DdZLsuOrg1V89x6qf+3xdL79S3ak6UX7Q98jvau/VN1K/S+NSfHd+tY/2oy2+Vr2369vx9VP1mPKQ9LtVX9QvtI+tePLkuvr/+rvEVV8sPVB9rtqj49A6KU8uHq04Ux2xdCfVE+IlqstXPMl/xQMUr9IVV39E9lq671p/2bm8QHix9Czx5pfu19F8lP+0D71xdA49V7qC4kDrKx1RvHDpJsor0udVVy6+suqWtX9g6aDr7+ql//W54vuqt5Vnb3WbtZ7qH8kuS1dY/ER8a+G7nqs+4NJXlj/In8UTV39N/Q7plf3U+gjvVZ/I7tIBZF/pO9I1xYtXHaw66qV4Lj689meJv0hXuo2L1lPNex2H/HXx4tXn1vw6T/HF4oTqs85XeUX6qnBX+wzlT1qPxZcUb6qvyjdXHbN0qeVPHb/qIuGY6g3hZ78XDgoPtf5aL8WFcE7je/L9OcTnlp6+dKSlR1cvrV8v/nJbH4oviG8Kx2UX9flUF0nH7vyeXKc6Unlb+lJ5j/YNan7F7XOoHl34s+wi/1e+k/658n79UX10xXvHo/5A+fTSdbre9a+lMxbH5ffKw/LnhTvShW77OKs/dVsv165LH+75+n/rne9L+bXwrH4j/r10lYXLsvOqd/Tc+r/wZNUXynPqQy2eJ9yXv4i3qN+iT/GT1ZdbOlivO0dxS/mp41I9vPBZ+0HEJ1Z9tPRy7c9QXMv/+vv6u53mZeHyOVYdtvBt1dvijYqPVT8uXqq+av1t9belj5yjfnGr36luaDx3fVef4xzSHdVP1r6d3i9cUL9c66a/33nyvfpBq+6Q/6ruUb7tuIR3et4t/5W9xIvWfoqui/a1dD16fZ93jpXH1F9T3CuvKv5Vz4p3q85d+V1+J7+RP0gHlZ8vHqB9Kn3Pbf5bdr+tr9WPFs9b9ZLwQnlCuopwSflbfLvPVVxofZeuo3pN9dviY4oX3a+/i1l1sPxV/iR/6PPVz1GcLB1K+qj4lXRG9cPE12/rmrVe+nxyLl1w/V20cFT5bdW74r/SXXS9+mLiEYqH6otrv3LX77aPrX6y/LH+o76H1kv1t/7uQn+Hdg71c5sXFY/FV/H7VTfLvtp/pLyu/Szik0+u6/jU51t1ya2/qT5d/ZLV15BuvPRB5ZXb/xOs+kd98FVv9371lZSPivuKv/Jv8SDlwaU/at7q42idxd/6eQ7xWeUx6aSrfln+81Lev/rOi78qXqU3dN2kx0jv0v4Q1fkrL68+zsJX7Ru57UMJh2/vk50Wn1ffQby6n+dQ3pJ9peeJtwkHiserT6r51F6qkxrv/V5xLP4sfVp8oPWDeN2qd8X7yuOUJ5beu/BHfai1T6njUD0nHaTjVV3fuGgc6X1d53No3rKLnnury7zUP1Z9oPwr/1x4rPdpfTrO1b9Rn07zlT8JJ8T3bvPO6v+vdVHfSPWExiHeVv8R7ms9+7v0Kf1dhXj94jPlncJb6QBLX+g4Vf+udb2Ng1v+tvZ5qP7R/JXv1bcRn5PuJdxYfUnVKU++X/xKfE68Qv6w5iH8Wjy+zxUffXKddLSlC8qvhBPLPrd919UvUN2teqX4Iz9ZuKh9BYoT2e/2/3TL/xbeqF8jPLrdr7PwRP23W129+VV6h/RV8eyX6p7SKcRjNF75gfrK0lX6/nMoXwgnFo/Susu/Vz315Hf1R7WesuvqD0oP6HorLqX7qV4VPq5+j/pdnb90Q+mUvV/XKe9pP43yWs/VBxR+dF0VH9LNzqHrxUOWXzd+629dz/J/6d8rHwr/9RzVIytOtb7y5/qj9Fnhr/Rp8cNVZ2re9ae+T/OUbtv71P8UPxJ+LR2567VwsLituBEe1h9u66HWZZqX+J3y4spPS4+QvetH0vHWOgsftB7iu0tvXH2d3ic9T7xa+pLq5JU3tA9FeCzeo3hTHVn+uPpMqy96qxPVn1Sv3+bxcyx87HukJ659FMrPyk+ap/L3wj3pqOWFt3lC8Vg7K87k/7KzeLrwf/Vplu53m8fUF+14z7H2XUmnVn4RjxH+y07KNwtHxMdlH/XZlC+Fy6vPX5ys36ke7H2ym/iI+LPiXXEm/6+e0vVddenSF27rLMWPeKTsqfVZ+4wUD9XPhZPitYqbFd9rP5DqJu27ES6qzlV+V/6WPV7KB1XPSZdafiPdQfFevFl16C2OKd8uvF37CFafQ/zztm/ZeJbdlZ8Vr1rnc5RH1E+X7i1/1P4O2VG8YPnfk+vkD/Iz6brqmwiH1G9p3VR7K48XZ5Y/SddYeav5Q3V08UB1jXTh5sHFHxV/wlmNc+GU9CLhl+qGtS9KfEU4Jp1TuuU5lE8UB8oPC197X3ntk+vUf5A+K51irad0hDU/5Rf9/dGyi+pI8aDFT8Uv5Q/Sr1TPqS+mPLfwX3rKym/iWcp/T85v71v7CMTbxMOenN/aUflm6cfNK1p/xZH8T3EhHiWeJX9Tf1C6W+NS41C+EG4Xh3u9dFHhwOoPFB+XHiy7rnyx/Gz5ufjN4uXSO+uHT85XXab6vXyjdemqS1b8qX+w/E74vfTUpW+pv7T0bOnLjVPxNK3LyguKv9t9StLxxdv0u+xUfz2H+rjqy0vvl651u+9BeprWQf0b8U3NS/3O1U+U3qL6s/lbebj+oevKg1X/1S7Fj6VT1N6tt8sXur7KF8p3un7p7s0jtbt0XvVHOi/1+1Z/QvipfvT6P0Jdf9m7dtB+3aUjVMdVPSOdvHaSfig7ab2WXqu6sP629heovrjVrcVzznvPsfKN/EB4v/qgnZ/wRjgmnUs8vPOXfiicva3HFMfnkA6rfFQ7qY/fdV/7Tc6xeJr0L63bstuTc9UfWpfFI5dfql5SnSO9duUp+b/8Z+myer7GL52089a6Ka7Oc/8DUEsBAi0DLQAAAAgAAAAhACurbKnIBgAAkCcAAAcAAAAAAAAAAAAAAIABAAAAAHNlbC5ucHlQSwECLQMtAAAACAAAACEAMhz6nUgSAACwzgAABQAAAAAAAAAAAAAAgAEBBwAAeS5ucHlQSwUGAAAAAAIAAgBoAAAAgBkAAAAA'))
import numpy as np; z=np.load('/kaggle/working/out/pl.npz'); print('pseudo-labels', z['sel'].sum())

In [ ]:
import glob, os, subprocess, time
os.chdir('/kaggle/working')
D = os.path.dirname(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)[0]); print('DATA', D)
!pip -q install timm 2>/dev/null
EPOCHS = 30
p0 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 UP=2 BS=64 SEED_OFF=19000 TAG=effb2_up2s5 P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb2 PL > out/log_effb2_up2s5.txt 2>&1', shell=True)
p1 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 UP=2 BS=64 SEED_OFF=19500 TAG=effb0_up2s5 P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} 30 1 timm_effb0 PL > out/log_effb0_up2s5.txt 2>&1', shell=True)
print('started')

In [ ]:
import time
while p0.poll() is None or p1.poll() is None:
    for f in ['out/log_effb2_up2s5.txt', 'out/log_effb0_up2s5.txt']:
        try:
            lines = [l for l in open(f).read().splitlines() if 'fold' in l or '==' in l or 'Error' in l or 'per class' in l]
            print(f, '|', *lines[-3:], sep='\n  ')
        except FileNotFoundError:
            print(f, 'not created yet')
    print('-' * 60, time.strftime('%H:%M:%S'), flush=True); time.sleep(300)
print('BOTH DONE')

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile('r2_outputs_run13_up2_C.zip', 'w') as z:
    for f in glob.glob('out/*.npy') + glob.glob('out/*.txt'): z.write(f)
FileLink('r2_outputs_run13_up2_C.zip')